# ♻️ WasteVision Studio — 02: OpenCV Preprocessing & 3D Edge Surfaces
### استكشاف ومعالجة صور النفايات ومحاكاة سطح التدرج ثلاثي الأبعاد
يستعرض هذا الدفتر قراءة الصور بـ OpenCV، مقارنة فضاءات الألوان RGB و HSV و LAB، تطبيق مرشح التباين الموضعي المتكيف CLAHE، ورسم سطح الحواف ثلاثي الأبعاد.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D

data_dir = Path("../data/dataset/dataset-resized")
if not data_dir.exists(): data_dir = Path("data/dataset/dataset-resized")

img_files = list(data_dir.rglob("*.jpg"))
print(f"إجمالي الصور المتوفرة للاستكشاف: {len(img_files)}")


In [ ]:
# 1. مقارنة فضاءات الألوان ومرشح التباين CLAHE
sample_path = img_files[0] if img_files else None
if sample_path:
    bgr = cv2.imread(str(sample_path))
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2RGB)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb); axes[0].set_title("RGB Original"); axes[0].axis("off")
    axes[1].imshow(hsv[:, :, 0], cmap="hsv"); axes[1].set_title("HSV - Hue Channel"); axes[1].axis("off")
    axes[2].imshow(enhanced); axes[2].set_title("CLAHE Enhanced"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# 2. محاكاة سطح الحواف ثلاثي الأبعاد (OpenCV Sobel 3D Surface)
if sample_path:
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, (80, 80))
    gx = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(small, cv2.CV_32F, 0, 1, ksize=3)
    mag = cv2.magnitude(gx, gy)

    # Matplotlib 3D Surface
    fig = plt.figure(figsize=(9, 5))
    ax = fig.add_subplot(111, projection='3d')
    X, Y = np.meshgrid(range(small.shape[1]), range(small.shape[0]))
    surf = ax.plot_surface(X, Y, mag, cmap='viridis', edgecolor='none')
    fig.colorbar(surf, shrink=0.5, aspect=8)
    ax.set_title("OpenCV Sobel 3D Edge Magnitude Surface")
    plt.show()
